# 01 · Exploración y Preparación de Datos — Andina Crédito

**Objetivo:** modelo de probabilidad de *default* (mora 90+ días a 12 meses).

Contenido:
1. Exploración estructural.
2. Target.
3. Calidad de datos (problemas + decisiones).
4. Limpieza (`src/prepare.py`).
5. **Análisis de correlaciones** (tabla + matriz interactiva + ranking).
6. **Señal predictiva del Top 10** (por quintiles si es numérica / por categoría si es categórica).
7. **Análisis del pricing** (`tasa_interes_anual`).
8. Análisis temporal (define la validación).

> Gráficos **interactivos con Plotly** · paleta corporativa **Banco BICE**.
> Limpieza centralizada en `src/prepare.py` (fit/transform, sin *data leakage*).


In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.metrics import roc_auc_score

pio.renderers.default = 'notebook'

# ---- Paleta corporativa Banco BICE ----
BICE_AZUL       = '#0E162A'   # azul marino (principal)
BICE_AZUL_MED   = '#2E536D'   # azul medio
BICE_AZUL_CLARO = '#A9BDDF'   # azul claro
BICE_ACENTO     = '#CE894D'   # terracota (acento / destacado)
BICE_PETROLEO   = '#062C33'   # azul petróleo (contraste)
PALETA      = [BICE_AZUL, BICE_ACENTO]                 # 2 categorías
ESCALA_BICE = ['#A9BDDF', '#2E536D', '#0E162A']        # escala continua claro→oscuro

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
print('Train:', train.shape, '| Test:', test.shape)

## 1. Exploración estructural

In [ ]:
print('Columnas solo en train:', set(train.columns) - set(test.columns))
train.dtypes

In [ ]:
train.head()

## 2. Target: `default_12m`

In [ ]:
dist = train['default_12m'].value_counts().sort_index()
tasa = train['default_12m'].mean()*100
print(dist); print(f"\nTasa de default: {tasa:.2f}%")

fig = px.bar(x=['No default (0)','Default (1)'], y=dist.values, text=dist.values,
             color=['No default (0)','Default (1)'], color_discrete_sequence=PALETA,
             title=f'Distribución del target · tasa de default = {tasa:.2f}%')
fig.update_layout(showlegend=False, xaxis_title='', yaxis_title='N° solicitudes')
fig.update_traces(textposition='outside'); fig.show()

**Hallazgo:** ~9,9% de default → desbalance. Usaremos **AUC-ROC, KS, PR-AUC** (no *accuracy*).

## 3. Calidad de datos — problemas detectados
### 3.1 Valores nulos

In [ ]:
nulos = pd.DataFrame({'nulos_train': train.isnull().sum(),
    'pct_train': (train.isnull().mean()*100).round(1),
    'pct_test': (test.isnull().mean()*100).round(1)})
nulos = nulos[nulos['nulos_train'] > 0]; print(nulos)

fig = px.bar(nulos.reset_index(), x='index', y='pct_train', text='pct_train',
             color_discrete_sequence=[BICE_AZUL_MED], title='% de valores nulos por columna (train)')
fig.update_layout(xaxis_title='', yaxis_title='% nulos')
fig.update_traces(textposition='outside', texttemplate='%{text}%'); fig.show()

**Hallazgo:** `ingreso_declarado` (~18%) y `antiguedad_laboral_meses` (~16%).
**Decisión:** imputar mediana (solo train) + **bandera de faltante**.

### 3.2 Rangos imposibles — edad

In [ ]:
fig = px.histogram(train, x='edad', nbins=60, color_discrete_sequence=[BICE_AZUL],
                   title='Distribución de edad (con outliers)')
fig.add_vline(x=100, line_dash='dash', line_color=BICE_ACENTO,
              annotation_text='límite 100 años', annotation_position='top')
fig.update_layout(xaxis_title='edad', yaxis_title='frecuencia', bargap=0.02); fig.show()
print('Edad > 100:', (train['edad'] > 100).sum(), 'filas (hasta', int(train['edad'].max()), 'años)')

**Hallazgo:** 70 filas con edad ≥ 100 (hasta 133). **Decisión:** → nulo → imputar + `flag_edad_invalida`.

### 3.3 Inconsistencia de unidades — ingreso (hallazgo principal)

In [ ]:
bajos = train[train['ingreso_declarado'] < 50000]['ingreso_declarado']
print(f"Filas con ingreso < 50.000 CLP: {len(bajos):,} (~{len(bajos)/len(train)*100:.0f}%)")
print(f"Máx del grupo bajo: {bajos.max():,.0f}")
ing = train['ingreso_declarado'].dropna()
print(f"Valores entre 5.014 y 280.000: {((ing>5014)&(ing<280000)).sum()}  <- 0 = gap limpio")

**Hallazgo:** ~10% con ingreso en 280–5.014 y **gap perfecto**. Es error de unidad (miles de pesos).
**Decisión (Opción A):** ×1000 + `flag_ingreso_corregido` (supuesto a validar con negocio).

### 3.4 Duplicados y categóricas

In [ ]:
print('id duplicados:', train['id_solicitud'].duplicated().sum(),
      '| filas duplicadas:', train.duplicated().sum())
for c in ['tipo_empleo','region','canal','dia_semana_solicitud']:
    print(f'  {c}: {train[c].nunique()} valores únicos')

**Hallazgo:** sin duplicados; categóricas limpias.

## 4. Aplicar limpieza (`src/prepare.py`)

In [ ]:
from prepare import preparar_datos
train_clean, test_clean, params = preparar_datos(train, test)
print('Parámetros (de train):', params)
print('Nulos restantes:', train_clean[['edad','ingreso_declarado','antiguedad_laboral_meses']].isna().sum().to_dict())
print('Banderas:', [c for c in train_clean.columns if c.startswith('flag_')])

## 5. Análisis de correlaciones ⭐

Primero medimos relaciones lineales entre variables numéricas y con el target.
Este ranking nos dirá **qué variables analizar en detalle** en la sección 6.

In [ ]:
num_cols = ['edad','ingreso_declarado','antiguedad_laboral_meses','antiguedad_cliente_meses',
            'score_buro','deuda_sistema','num_creditos_vigentes','peor_morosidad_12m',
            'num_consultas_buro_3m','num_contactos_ult_trimestre','uso_linea_credito_pct',
            'monto_solicitado','plazo_meses','tasa_interes_anual','ratio_deuda_ingreso',
            'flag_ingreso_corregido','flag_edad_invalida',
            'flag_ingreso_declarado_faltante','flag_antiguedad_laboral_meses_faltante','default_12m']
corr = train_clean[num_cols].corr()

### 5.1 Tabla completa de correlaciones con el target

In [ ]:
corr_target = corr['default_12m'].drop('default_12m').sort_values(ascending=False)
tabla_corr = corr_target.to_frame('correlacion_con_default')
tabla_corr['direccion'] = np.where(tabla_corr['correlacion_con_default']>=0, '↑ aumenta riesgo', '↓ reduce riesgo')
(tabla_corr.style
   .background_gradient(cmap='RdYlGn_r', subset=['correlacion_con_default'])
   .format({'correlacion_con_default':'{:+.3f}'})
   .set_caption('Correlación de cada variable con default_12m (ordenada)'))

### 5.2 Matriz de correlación interactiva

In [ ]:
fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Matriz de correlación (variables numéricas)')
fig.update_layout(width=900, height=800, xaxis_tickangle=-45, coloraxis_colorbar_title='corr')
fig.update_xaxes(tickfont=dict(size=10)); fig.update_yaxes(tickfont=dict(size=10)); fig.show()

### 5.3 Ranking de correlación con el target

In [ ]:
rank = corr_target.reindex(corr_target.abs().sort_values(ascending=True).index)
colores = ['#C00000' if v>=0 else '#2E7D32' for v in rank.values]
fig = go.Figure(go.Bar(x=rank.values, y=rank.index, orientation='h', marker_color=colores,
    text=[f'{v:+.3f}' for v in rank.values], textposition='outside'))
fig.update_layout(title='Ranking de correlación con default_12m (por |correlación|)',
    xaxis_title='correlación con el target', yaxis_title='', height=650, margin=dict(l=230))
fig.add_vline(x=0, line_color='#888'); fig.show()

**Nota:** la correlación de Pearson solo aplica a variables numéricas. Para incluir
también las **categóricas** (`tipo_empleo`, `region`, `canal`, `dia_semana`) en la selección
de la sección 6, usamos una medida univariada comparable (AUC) más abajo.

## 6. Señal predictiva — Top 10 variables

Para rankear **todas** las variables (numéricas y categóricas) con una métrica comparable,
usamos el **AUC univariado** contra el target:
- **Numéricas:** AUC directo del valor (se toma `max(auc, 1-auc)` para capturar dirección).
- **Categóricas:** se codifica cada categoría por su tasa de default (solo train) y se calcula el AUC.

Luego, para el **Top 10**, mostramos la **tasa de default** por *quintil* (numéricas) o por
*categoría* (categóricas/banderas). Esto revela relaciones **no lineales** que la correlación no capta.

In [ ]:
y = train_clean['default_12m']
cat_feats = ['tipo_empleo','region','canal','dia_semana_solicitud']
num_feats = [c for c in num_cols if c != 'default_12m']

strength = {}
for c in num_feats:
    try:
        a = roc_auc_score(y, train_clean[c]); strength[c] = max(a, 1-a)
    except Exception:
        pass
for c in cat_feats:
    rate = train_clean.groupby(c)['default_12m'].transform('mean')
    a = roc_auc_score(y, rate); strength[c] = max(a, 1-a)

rank_auc = pd.Series(strength).sort_values(ascending=False)
top10 = rank_auc.head(10)
print('TOP 10 variables por AUC univariado:')
print(top10.round(3))

fig = px.bar(top10[::-1], orientation='h', color=top10[::-1].values,
             color_continuous_scale=ESCALA_BICE,
             title='Top 10 variables por poder predictivo univariado (AUC)')
fig.update_layout(xaxis_title='AUC univariado', yaxis_title='', height=520,
                  coloraxis_showscale=False, margin=dict(l=230))
fig.update_traces(texttemplate='%{x:.3f}', textposition='outside'); fig.show()

### 6.1 Tasa de default por quintil / categoría (Top 10)

Función auxiliar: agrupa por quintil (numéricas con >6 valores) o por categoría, y calcula
la tasa de default y el volumen de cada grupo.

In [ ]:
def señal(var, q=5):
    df = train_clean
    es_cat = (df[var].dtype == object) or (df[var].nunique() <= 6)
    if es_cat:
        g = df.groupby(var)['default_12m'].agg(tasa='mean', n='count').reset_index()
        g[var] = g[var].astype(str); etiqueta = var
    else:
        b = pd.qcut(df[var], q, duplicates='drop')
        g = df.groupby(b, observed=True)['default_12m'].agg(tasa='mean', n='count').reset_index()
        g[var] = g[var].astype(str); etiqueta = f'{var} (quintil)'
    return g, etiqueta, es_cat

# Grilla 5x2 con la señal de cada variable del Top 10
top_vars = list(top10.index)
fig = make_subplots(rows=5, cols=2, subplot_titles=top_vars, vertical_spacing=0.06, horizontal_spacing=0.12)
for i, var in enumerate(top_vars):
    g, etiqueta, es_cat = señal(var)
    r, c = i//2 + 1, i%2 + 1
    fig.add_trace(go.Bar(x=g[var], y=g['tasa'], marker_color=BICE_AZUL_MED,
                         customdata=g['n'],
                         hovertemplate='%{x}<br>tasa=%{y:.1%}<br>n=%{customdata}<extra></extra>',
                         showlegend=False), row=r, col=c)
    fig.add_hline(y=y.mean(), line_dash='dot', line_color=BICE_ACENTO, row=r, col=c)
fig.update_layout(height=1400, title_text='Tasa de default por quintil/categoría · Top 10 (línea = tasa global)')
fig.update_yaxes(tickformat='.0%')
fig.update_xaxes(tickangle=-30, tickfont=dict(size=8))
fig.show()

**Cómo leerlo:** la línea punteada (terracota) es la tasa de default global (~9,9%).
Barras muy por encima/por debajo indican **fuerte poder discriminante**. Un patrón
**monótono** (sube o baja de forma consistente por quintil) es señal limpia y estable;
un patrón en "U" indica no linealidad que los modelos de árboles capturarán mejor que uno lineal.

## 7. Análisis del pricing — `tasa_interes_anual` ⭐

La tasa **la asigna el sistema de pricing de Andina** (risk-based pricing): a mayor riesgo
percibido, mayor tasa. Por eso merece un análisis aparte: es predictiva, **pero no es señal
independiente** — codifica el juicio de riesgo que ya hizo otro sistema.

In [ ]:
print('Correlaciones clave de la tasa:')
print(f"  tasa vs score_buro : {train_clean['tasa_interes_anual'].corr(train_clean['score_buro']):+.3f}")
print(f"  tasa vs default    : {train_clean['tasa_interes_anual'].corr(train_clean['default_12m']):+.3f}")
print(f"  score vs default   : {train_clean['score_buro'].corr(train_clean['default_12m']):+.3f}")

### 7.1 Tasa de default por quintil de `tasa_interes_anual`

In [ ]:
b = pd.qcut(train_clean['tasa_interes_anual'], 5)
g = train_clean.groupby(b, observed=True)['default_12m'].agg(tasa='mean', n='count').reset_index()
g['tasa_interes_anual'] = g['tasa_interes_anual'].astype(str)
fig = px.bar(g, x='tasa_interes_anual', y='tasa', text='tasa',
             color='tasa', color_continuous_scale=ESCALA_BICE,
             title='Tasa de default por quintil de tasa_interes_anual')
fig.add_hline(y=train_clean['default_12m'].mean(), line_dash='dot', line_color=BICE_ACENTO,
              annotation_text='tasa global')
fig.update_layout(xaxis_title='quintil de tasa asignada', yaxis_title='tasa de default',
                  coloraxis_showscale=False, yaxis_tickformat='.0%')
fig.update_traces(texttemplate='%{text:.1%}', textposition='outside'); fig.show()

### 7.2 Relación tasa ↔ score (evidencia del pricing basado en riesgo)

Si el pricing se basa en riesgo, debe existir una relación **negativa clara**: peor score → mayor tasa.

In [ ]:
m = train_clean.sample(min(5000, len(train_clean)), random_state=1)
fig = px.scatter(m, x='score_buro', y='tasa_interes_anual',
                 color='default_12m', color_continuous_scale=[BICE_AZUL_CLARO, '#C00000'],
                 opacity=0.5, title='tasa_interes_anual vs score_buro (color = default)')
fig.update_layout(xaxis_title='score_buro', yaxis_title='tasa asignada (%)',
                  coloraxis_colorbar_title='default'); fig.show()

**Hallazgos del pricing:**
- La tasa correlaciona **≈ −0,56 con el score** → confirma **pricing basado en riesgo**.
- Correlaciona **≈ +0,21 con el default**: es predictiva, pero de forma **derivada** (vía score).
- **Implicancia de modelado:** entrenaremos **con y sin** `tasa_interes_anual`. Si el modelo
  depende demasiado de ella, se degradará cuando cambie la política de pricing. Reportar ambos
  demuestra criterio y protege el rendimiento en producción.

## 8. Análisis temporal (define la validación)

In [ ]:
train_clean['mes'] = pd.to_datetime(train_clean['fecha_solicitud']).dt.to_period('M').astype(str)
serie = train_clean.groupby('mes')['default_12m'].mean()
print(serie.round(3))
fig = px.line(serie.reset_index(), x='mes', y='default_12m', markers=True,
              color_discrete_sequence=[BICE_ACENTO], title='Tasa de default por mes de solicitud')
fig.update_layout(xaxis_title='mes', yaxis_title='tasa de default', xaxis_tickangle=-45,
                  yaxis_tickformat='.0%'); fig.show()

**Hallazgo CRÍTICO:** el default **crece** en el tiempo (7% → 13%). Un split aleatorio sería
optimista y deshonesto → usaremos **validación temporal (out-of-time)**.

## Resumen de decisiones (Paso 1)

| # | Problema | Decisión |
|---|---|---|
| 1 | Nulos ingreso (18%) / antigüedad (16%) | Imputar mediana (train) + bandera |
| 2 | Edad ≥ 100 (70 filas) | → nulo → imputar + bandera |
| 3 | Ingreso en miles (~10%, gap limpio) | ×1000 + bandera (Opción A) |
| 4 | Desbalance target (9,9%) | Métricas AUC/KS/PR-AUC |
| 5 | `tasa` refleja pricing por riesgo | Modelar con y sin la variable |
| 6 | Default creciente en el tiempo | Validación temporal (out-of-time) |

**Anti-leakage:** todos los estadísticos se calculan solo en train y se aplican a test.
